# Training Loops

In [ ]:
import pickle
import random
from cs336_basics.tokenizer import BPE_Tokenizer_Training, Tokenizer

### TinyStories tokenizer training

In [ ]:
def save_bpe_artifacts(vocab, merges, out_dir):
    import os
    os.makedirs(out_dir, exist_ok=True)

    with open(out_dir + "/vocab.pkl", "wb") as f:
        pickle.dump(vocab, f)

    with open(out_dir + "/merges.pkl", "wb") as f:
        pickle.dump(merges, f)

vocab, merges = BPE_Tokenizer_Training(
    input_path="../data/TinyStoriesV2-GPT4-train.txt",
    vocab_size=10000,
    special_tokens=["<|endoftext|>"],
    parallelize=True,
)

save_bpe_artifacts(vocab, merges, "tinystories_bpe_outputs")

Time at start: 262905.450477125
Time before pretok: 262905.450571708
Time after pretok: 262970.7299135
Time after merges: 263264.745225916


## OWT Training Loop

In [3]:
with open("../data/owt_bpe_32k/vocab.pkl", "rb") as f:
    vocab = pickle.load(f)
with open("../data/owt_bpe_32k/merges.pkl", "rb") as f:
    merges = pickle.load(f)

print(type(vocab), len(vocab))
print(type(merges), len(merges))
print(random.sample(list(vocab.items()), 10))
print(merges[:10])

<class 'dict'> 32000
<class 'list'> 31743
[(2630, b' pur'), (25743, b'imps'), (734, b'round'), (21376, b'uba'), (21262, b'vered'), (12069, b' advised'), (22564, b' merchant'), (31372, b' chimpanz'), (4920, b' Supreme'), (5599, b' considering')]
[(b' ', b't'), (b' ', b'a'), (b'h', b'e'), (b'i', b'n'), (b'r', b'e'), (b' t', b'he'), (b'o', b'n'), (b'e', b'r'), (b' ', b's'), (b' ', b'w')]


In [10]:
owt_tokenizer = Tokenizer(vocab, merges, special_tokens=["<|endoftext|>"])
print(f"Loaded tokenizer with {len(owt_tokenizer.vocab)} tokens")

Loaded tokenizer with 32000 tokens


## Tokenize OWT data to memmap files

In [ ]:
import numpy as np
import os

def tokenize_to_memmap(tokenizer, txt_path, out_path, buffer_size=1_000_000):
    """Tokenize a text file and stream token IDs directly to a raw binary file on disk.
    Never holds more than buffer_size tokens in memory at once.
    The resulting file can be loaded with np.memmap(..., dtype=np.uint16, mode='r')."""
    if os.path.exists(out_path):
        print(f"{out_path} already exists, skipping.")
        return

    print(f"Tokenizing {txt_path} -> {out_path}")
    total_tokens = 0
    with open(txt_path, "r", encoding="utf-8") as f_in, open(out_path, "wb") as f_out: 
        buffer = []
        for token_id in tokenizer.encode_iterable(f_in):
            buffer.append(token_id)
            if len(buffer) >= buffer_size:
                arr = np.array(buffer, dtype=np.uint16)
                arr.tofile(f_out)
                total_tokens += len(buffer)
                buffer.clear()
        # Flush remaining
        if buffer:
            arr = np.array(buffer, dtype=np.uint16)
            arr.tofile(f_out)
            total_tokens += len(buffer)

    print(f"Done. Total tokens: {total_tokens:,}")

# Tokenize train and valid sets
tokenize_to_memmap(owt_tokenizer, "../data/owt_train.txt", "../data/owt_train_tokens.bin")
tokenize_to_memmap(owt_tokenizer, "../data/owt_valid.txt", "../data/owt_valid_tokens.bin")

Tokenizing ../data/owt_train.txt -> ../data/owt_train_tokens.bin


## Hyperparameters

In [ ]:
import torch
from cs336_basics.transformer import Transformer_LM
from cs336_basics.training import (
    cross_entropy, learning_rate_schedule, gradient_clipping,
    AdamW, data_loading, save_checkpoint, load_checkpoint,
)
import wandb
import tqdm

# --- Device ---
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# --- Model hyperparameters ---
vocab_size = 32000
context_length = 256
num_layers = 4
d_model = 128
num_heads = 4
d_ff = 512
theta = 10000.0

# --- Training hyperparameters ---
batch_size = 32
max_lr = 1e-3
min_lr = 1e-4
warmup_iters = 500
max_iters = 5000
eval_interval = 250
eval_iters = 20
max_grad_norm = 1.0
weight_decay = 0.1
betas = (0.9, 0.95)

# --- Paths ---
checkpoint_dir = "../checkpoints/owt_small"
train_data_path = "../data/owt_train_tokens.bin"
valid_data_path = "../data/owt_valid_tokens.bin"
os.makedirs(checkpoint_dir, exist_ok=True)

## Load data with memmap

In [ ]:
# Memory-map the tokenized data — uses mmap under the hood so the full
# array is never loaded into RAM; pages are fetched lazily on access.
train_data = np.memmap(train_data_path, dtype=np.uint16, mode="r")
valid_data = np.memmap(valid_data_path, dtype=np.uint16, mode="r")
print(f"Train tokens: {len(train_data):,}")
print(f"Valid tokens: {len(valid_data):,}")

# Verify that all token IDs are within the expected vocabulary range
assert train_data.max() < vocab_size, f"Train data has token ID {train_data.max()} >= vocab_size {vocab_size}"
assert valid_data.max() < vocab_size, f"Valid data has token ID {valid_data.max()} >= vocab_size {vocab_size}"
print(f"All token IDs in range [0, {vocab_size})")
print(f"Train sample: {train_data[:20]}")
print(f"Valid sample: {valid_data[:20]}")

## Initialize model and optimizer

In [ ]:
model = Transformer_LM(
    vocab_size=vocab_size,
    context_length=context_length,
    num_layers=num_layers,
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    max_seq_len=context_length,
    theta=theta,
    device=device,
)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

optimizer = AdamW(model.parameters(), lr=max_lr, betas=betas, weight_decay=weight_decay)

## Evaluation helper

In [ ]:
@torch.no_grad()
def estimate_loss(model, train_data, valid_data, batch_size, context_length, device, eval_iters):
    """Estimate train and validation loss over eval_iters batches."""
    losses = {}
    for split_name, data in [("train", train_data), ("val", valid_data)]:
        total_loss = 0.0
        for _ in range(eval_iters):
            x, y = data_loading(data, batch_size, context_length, device)
            logits = model(x)
            loss = cross_entropy(logits, y)
            total_loss += loss.item()
        losses[split_name] = total_loss / eval_iters
    return losses

## Training loop

In [ ]:
# Initialize wandb
wandb.init(project="cs336-owt", config={
    "vocab_size": vocab_size,
    "context_length": context_length,
    "num_layers": num_layers,
    "d_model": d_model,
    "num_heads": num_heads,
    "d_ff": d_ff,
    "batch_size": batch_size,
    "max_lr": max_lr,
    "min_lr": min_lr,
    "warmup_iters": warmup_iters,
    "max_iters": max_iters,
    "max_grad_norm": max_grad_norm,
    "weight_decay": weight_decay,
})

for t in tqdm(range(1, max_iters + 1)):
    # Update learning rate
    lr = learning_rate_schedule(t, max_lr, min_lr, warmup_iters, max_iters)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    # Forward pass
    x, y = data_loading(train_data, batch_size, context_length, device)
    logits = model(x)
    loss = cross_entropy(logits, y)

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    gradient_clipping(model.parameters(), max_grad_norm)
    optimizer.step()

    # Log training loss every step
    wandb.log({"train/loss": loss.item(), "lr": lr}, step=t)

    # Periodic evaluation and checkpointing
    if t % eval_interval == 0 or t == max_iters:
        losses = estimate_loss(model, train_data, valid_data, batch_size, context_length, device, eval_iters)
        print(f"Step {t}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, lr {lr:.6f}")
        wandb.log({"val/loss": losses["val"], "eval/train_loss": losses["train"]}, step=t)

        # Save checkpoint
        ckpt_path = os.path.join(checkpoint_dir, f"step_{t}.pt")
        save_checkpoint(model, optimizer, t, ckpt_path)
        print(f"  Saved checkpoint to {ckpt_path}")

wandb.finish()
print("Training complete.")

## Generating Text